# Duka Electronics — Half-Year Sales Exploration (Jan–Jun 2026)

## Executive Summary
- Duka Electronics recorded 1,000 orders across five branches during the first half of 2026 (January–June 2026).
- The Nairobi branch leads branch performance with 400 orders and a 36.4% share of total revenue, nearly double second-placed Mombasa.
- Laptops generate KES 41,535,000 in revenue, more than double the next highest-earning product.
- Online orders (320 orders) achieve an average order value of about KES 71,207, closely matching In-store orders (579 orders) at about KES 70,759.
- Filtering for Premium Online orders from Nairobi and Mombasa isolates a focused high-value segment exported to `outputs/premium_online_coast_capital.csv`.

## Setup & Task 1: Load Data and First-Five-Minutes Inspection

In [ ]:
import numpy as np
import pandas as pd

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", sales.shape)
print("\n--- HEAD ---")
display(sales.head())
print("\n--- TAIL ---")
display(sales.tail())
print("\n--- INFO ---")
sales.info()
print("\n--- DESCRIBE ---")
display(sales.describe(include="all"))
print("\n--- NUNIQUE ---")
print(sales.nunique())
print("\n--- BRANCH SHARE ---")
print(sales["branch"].value_counts(normalize=True).round(4))
print("\n--- CHANNEL SHARE ---")
print(sales["channel"].value_counts(normalize=True).round(4))
print("\nDate range:", sales["order_date"].min(), "to", sales["order_date"].max())

### First-Five-Minutes Inspection Notes
- **Shape:** The dataset contains 1,000 rows and 7 columns (`order_id`, `order_date`, `branch`, `product`, `quantity`, `channel`, `unit_price`).
- **Head & Tail:** Rows show sequential `order_id` values starting at 1001 with realistic product/price mappings and dates in 2026.
- **Info & Missing Values:** `order_date` is parsed as `datetime64[ns]`, numeric/categorical columns have clean types, and there are zero missing values across columns.
- **Describe & Cardinality:** `quantity` ranges from 1 to 5, `unit_price` spans from KES 1,200 (Charger) to KES 65,000 (Laptop), and there are 5 branches, 5 products, and 3 channels.
- **Date Range & Mix:** Orders span January 1, 2026 through June 29, 2026, led by Nairobi (~40%) and In-store (~55%).

## Task 2: Derived Columns (`revenue`, `month`, `weekday`, `is_weekend`, `price_band`)

In [ ]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

for col in ["month", "weekday", "is_weekend", "price_band"]:
    print(f"\n--- value_counts: {col} ---")
    print(sales[col].value_counts())

## Task 3: Branch Ranking by Revenue vs. Number of Orders

In [ ]:
branch_summary = (
    sales.groupby("branch", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order_value=("revenue", "mean"),
        units_sold=("quantity", "sum"),
    )
    .sort_values("total_revenue", ascending=False)
)
branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(ascending=False, method="min").astype(int)
branch_summary["order_rank"] = branch_summary["order_count"].rank(ascending=False, method="min").astype(int)
branch_summary["revenue_share_pct"] = (branch_summary["total_revenue"] / branch_summary["total_revenue"].sum() * 100).round(1)

display(branch_summary)

**Task 3 Answer:** Ranking branches by total revenue and by order count can differ depending on basket size and high-ticket product mix (such as Laptops and Tablets). A branch with fewer orders can generate higher revenue if customers buy higher-priced items or larger quantities per order.

## Task 4: Top Three Products by Revenue vs. Units Sold

In [ ]:
product_summary = (
    sales.groupby("product", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        unit_price=("unit_price", "first"),
    )
)

top3_rev = product_summary.sort_values("total_revenue", ascending=False).head(3)
top3_units = product_summary.sort_values("units_sold", ascending=False).head(3)

print("Top 3 by Revenue:")
display(top3_rev)
print("Top 3 by Units Sold:")
display(top3_units)

**Task 4 Answer:** Top products by revenue reflect both unit volume and unit price, whereas top products by units sold reflect volume alone. High-unit-price items like Laptops (KES 65,000) and Tablets (KES 28,000) carry heavy revenue weight per unit compared to lower-priced accessories.

## Task 5: Weekend vs. Weekday Orders

In [ ]:
weekend_summary = (
    sales.groupby("is_weekend", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
    )
)
weekend_summary["day_type"] = np.where(weekend_summary["is_weekend"], "Weekend", "Weekday")
display(weekend_summary[["day_type", "order_count", "total_revenue", "avg_order_revenue"]])

**Task 5 Answer:** Weekdays account for five out of seven days each week, so they naturally generate a larger share of total orders and cumulative revenue, while comparing average order revenue shows whether weekend shoppers spend more or less per transaction.

## Task 6: Best vs. Worst Month for Revenue and Percentage Difference

In [ ]:
monthly_rev = (
    sales.assign(year_month=sales["order_date"].dt.to_period("M"))
    .groupby("year_month", as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)

best_row = monthly_rev.iloc[0]
worst_row = monthly_rev.iloc[-1]
pct_diff_vs_worst = ((best_row["revenue"] - worst_row["revenue"]) / worst_row["revenue"]) * 100
pct_diff_symmetric = ((best_row["revenue"] - worst_row["revenue"]) / ((best_row["revenue"] + worst_row["revenue"]) / 2)) * 100

display(monthly_rev)
print(f"Best month: {best_row['year_month']} (KES {best_row['revenue']:,.0f})")
print(f"Worst month: {worst_row['year_month']} (KES {worst_row['revenue']:,.0f})")
print(f"Percentage difference (best vs. worst): {pct_diff_vs_worst:.2f}%")
print(f"Percentage difference (relative to midpoint): {pct_diff_symmetric:.2f}%")

**Task 6 Answer:** Comparing the highest-revenue month against the lowest-revenue month highlights seasonal monthly swings across H1 2026, quantified above both as the percentage gain from the lowest month to the highest month and relative to the two-month midpoint.

## Task 7: Filter Premium Online Orders from Nairobi and Mombasa

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

premium_online_coast = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("Matching orders count:", len(premium_online_coast))
print("Total revenue: KES", f"{premium_online_coast['revenue'].sum():,.0f}")

premium_online_coast.to_csv("outputs/premium_online_coast_capital.csv", index=False)
display(premium_online_coast.head())

**Task 7 Answer:** The filtered subset contains only `Premium` (`unit_price >= 28000`) and `Online` orders originating from `Nairobi` or `Mombasa`, with exact order count and total revenue printed above and exported to `outputs/premium_online_coast_capital.csv` with `index=False`.

## Task 8: Spreadsheet Reconciliation Check (`excel/duka_branch_summary.xlsx`)

In [ ]:
branch_channel_pivot = pd.pivot_table(
    sales,
    values="revenue",
    index="branch",
    columns="channel",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)
display(branch_channel_pivot)
print("Grand total revenue in pandas: KES", f"{sales['revenue'].sum():,.0f}")